# Model: RT-DETR (Real-Time Detection Transformer)

End-to-end transformer detector with no NMS and real-time inference speed.

**Architecture**: ResNet-101-D backbone → Hybrid Encoder (CNN + cross-scale attention) → Transformer decoder (300 queries)

**Framework**: Ultralytics (pure PyTorch, no mmcv)

**Training**: AdamW + cosine LR, 100 epochs, fine-tuned from COCO pretrained weights

**Key advantage**: NMS-free inference, strong small-object recall, simple one-line install

## Installation

Run once. Only requires `ultralytics` — no mmcv or MMDetection.

In [1]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics', 'pyyaml', '-q'])
print('Installation complete.')

Installation complete.


## GPU Verification & Imports

In [2]:
import torch
import os, json, time, yaml
import numpy as np, cv2
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from ultralytics import RTDETR

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\User\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.0 GB


## Paths & Constants

In [ ]:
BASE_DIR    = 'C:/Users/User/Desktop/Ipynb'
DATASET_DIR = os.path.join(BASE_DIR, 'archive', 'dataset_v2')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'runs', 'rtdetr')
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CLASSES    = 3
CONF_THRESHOLD = 0.5
CLASS_NAMES    = ['Recyclable', 'Non-recyclable', 'Other']

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Output : {OUTPUT_DIR}')
print(f'Device : {DEVICE}')

## Convert COCO Annotations → YOLO Format

Ultralytics expects one `.txt` label file per image with lines:
`class_id  cx  cy  w  h`  (all values normalised 0–1).

Labels are written to `dataset_v2/labels/{train,val,test}/`.

In [4]:
def coco_to_yolo(ann_file, label_dir, split):
    os.makedirs(label_dir, exist_ok=True)
    with open(ann_file) as f:
        data = json.load(f)

    img_info   = {img['id']: img for img in data['images']}
    anns_by_img = defaultdict(list)
    for ann in data['annotations']:
        anns_by_img[ann['image_id']].append(ann)

    for img_id, img in img_info.items():
        iw, ih = img['width'], img['height']
        stem   = Path(img['file_name']).stem
        lines  = []
        for ann in anns_by_img[img_id]:
            x, y, bw, bh = ann['bbox']
            if bw <= 0 or bh <= 0:
                continue
            cx = max(0.0, min(1.0, (x + bw / 2) / iw))
            cy = max(0.0, min(1.0, (y + bh / 2) / ih))
            nw = max(0.0, min(1.0, bw / iw))
            nh = max(0.0, min(1.0, bh / ih))
            cat = ann['category_id'] - 1  # COCO 1-indexed → YOLO 0-indexed
            lines.append(f'{cat} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        with open(os.path.join(label_dir, stem + '.txt'), 'w') as f:
            f.write('\n'.join(lines))

    print(f'  {split}: {len(img_info)} images → {label_dir}')

print('Converting annotations...')
for split in ['train', 'val', 'test']:
    coco_to_yolo(
        ann_file  = os.path.join(DATASET_DIR, f'{split}_annotations.json'),
        label_dir = os.path.join(DATASET_DIR, 'labels', split),
        split     = split,
    )
print('Done.')

Converting annotations...
  train: 1050 images → C:/Users/User/Desktop/Ipynb\archive\dataset_v2\labels\train
  val: 300 images → C:/Users/User/Desktop/Ipynb\archive\dataset_v2\labels\val
  test: 150 images → C:/Users/User/Desktop/Ipynb\archive\dataset_v2\labels\test
Done.


## Dataset YAML

Tells Ultralytics where the images and labels are.

In [ ]:
yaml_path = os.path.join(BASE_DIR, 'taco_rtdetr.yaml')
dataset_cfg = {
    'path' : DATASET_DIR.replace('\\', '/'),
    'train': 'images/train',
    'val'  : 'images/val',
    'test' : 'images/test',
    'nc'   : NUM_CLASSES,
    'names': CLASS_NAMES,
}
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False)
print(f'Dataset YAML: {yaml_path}')

## Training

Fine-tunes `rtdetr-l.pt` (pretrained on COCO) on the TACO dataset.

- **Optimizer**: AdamW lr=1e-4 with cosine decay
- **Schedule**: 3-epoch linear warmup → cosine annealing to lr×0.01
- **Checkpointing**: `best_rtdetr.pt` saved by val mAP@50:95


In [ ]:
import shutil

model = RTDETR('rtdetr-l.pt')

model.train(
    data         = yaml_path,
    epochs       = 100,
    imgsz        = 640,
    batch        = 4,
    device       = DEVICE,
    project      = OUTPUT_DIR,
    name         = 'train',
    optimizer    = 'AdamW',
    lr0          = 1e-4,
    lrf          = 0.01,
    warmup_epochs= 3,
    cos_lr       = True,
    workers      = 0,
    save         = True,
    save_period  = 10,
    exist_ok     = True,
)

# Rename best.pt → best_rtdetr.pt to avoid confusion with other models
weights_dir = os.path.join(OUTPUT_DIR, 'train', 'weights')
src = os.path.join(weights_dir, 'best.pt')
dst = os.path.join(weights_dir, 'best_rtdetr.pt')
shutil.copy2(src, dst)
print(f'Saved best weights to: {dst}')


## Test Evaluation

Loads `best_rtdetr.pt`, runs inference on the test set, and reports:
- COCO mAP@50 and mAP@50-95 (via pycocotools)
- Counting MAE, within ±1, within ±3
- Average inference time

Ultralytics `predict()` returns boxes in the **original image coordinate space** automatically.


In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

best_ckpt = os.path.join(OUTPUT_DIR, 'train', 'weights', 'best_rtdetr.pt')
if not os.path.exists(best_ckpt):
    raise FileNotFoundError(f'No checkpoint found at {best_ckpt} — run training first.')

model = RTDETR(best_ckpt)
print(f'Loaded: {best_ckpt}')


In [8]:
with open(os.path.join(DATASET_DIR, 'test_annotations.json')) as f:
    test_coco_data = json.load(f)

gt_counts = defaultdict(int)
for ann in test_coco_data['annotations']:
    gt_counts[ann['image_id']] += 1

test_img_dir      = os.path.join(DATASET_DIR, 'images', 'test')
coco_preds        = []
count_errors      = []
inference_times   = []
per_image_results = []

for img_info in tqdm(test_coco_data['images'], desc='Test inference'):
    img_path = os.path.join(test_img_dir, img_info['file_name'])
    if not os.path.exists(img_path):
        continue

    t0      = time.perf_counter()
    results = model.predict(img_path, conf=CONF_THRESHOLD, verbose=False)
    inf_ms  = (time.perf_counter() - t0) * 1000
    inference_times.append(inf_ms)

    boxes_obj  = results[0].boxes
    img_id     = img_info['id']
    pred_count = len(boxes_obj)
    gt_count   = gt_counts.get(img_id, 0)
    count_errors.append(abs(pred_count - gt_count))

    per_image_results.append({
        'file':         img_info['file_name'],
        'gt_count':     gt_count,
        'pred_count':   pred_count,
        'count_error':  abs(pred_count - gt_count),
        'inference_ms': inf_ms,
    })

    for box in boxes_obj:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        score = float(box.conf[0])
        label = int(box.cls[0])
        coco_preds.append({
            'image_id':    img_id,
            'category_id': label + 1,
            'bbox':        [x1, y1, x2 - x1, y2 - y1],
            'score':       score,
        })

# =========================
# COCO mAP
# =========================
gt_coco = COCO()
gt_coco.dataset = test_coco_data
gt_coco.createIndex()

box_map50 = box_map50_95 = 0.0
if coco_preds:
    dt_coco      = gt_coco.loadRes(coco_preds)
    ev           = COCOeval(gt_coco, dt_coco, 'bbox')
    ev.evaluate(); ev.accumulate(); ev.summarize()
    box_map50_95 = float(ev.stats[0])
    box_map50    = float(ev.stats[1])
else:
    print('No predictions above threshold.')

# =========================
# COUNTING METRICS
# =========================
within_1 = sum(1 for e in count_errors if e <= 1) / len(count_errors) * 100
within_3 = sum(1 for e in count_errors if e <= 3) / len(count_errors) * 100
avg_inf  = float(np.mean(inference_times[1:]) if len(inference_times) > 1 else inference_times[0])

print('\n' + '=' * 50)
print('RT-DETR TEST RESULTS')
print('=' * 50)
print(f'Box mAP@50:            {box_map50:.4f}')
print(f'Box mAP@50-95:         {box_map50_95:.4f}')
print(f'Counting MAE:          {np.mean(count_errors):.2f}')
print(f'Counting within +/-1:  {within_1:.1f}%')
print(f'Counting within +/-3:  {within_3:.1f}%')
print(f'Avg inference time:    {avg_inf:.1f} ms')
print(f'Median inference time: {np.median(inference_times):.1f} ms')
print('=' * 50)

Test inference: 100%|████████████████████████████████████████████████████████████████| 150/150 [00:17<00:00,  8.36it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.11s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.115
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.144
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.124
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.008
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.042
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.149
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.125
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.153
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.154
 Average Recall     (AR) @[ IoU=0.5

## Save Results

In [9]:
results_data = {
    'metrics': {
        'box_map50':         box_map50,
        'box_map50_95':      box_map50_95,
        'mask_map50':        0.0,
        'mask_map50_95':     0.0,
        'counting_mae':      float(np.mean(count_errors)),
        'counting_within_1': within_1,
        'counting_within_3': within_3,
        'avg_inference_ms':  avg_inf,
    },
    'per_image': per_image_results,
}

out_path = os.path.join(OUTPUT_DIR, 'rtdetr_results.json')
with open(out_path, 'w') as f:
    json.dump(results_data, f, indent=2)
print(f'Results saved to: {out_path}')

Results saved to: C:/Users/User/Desktop/Ipynb\runs\rtdetr\rtdetr_results.json


## Visualize Predictions

In [ ]:
COLORS = plt.colormaps['tab10']

sample_infos = [
    i for i in test_coco_data['images']
    if os.path.exists(os.path.join(test_img_dir, i['file_name']))
][:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, img_info in zip(axes.flat, sample_infos):
    img_path  = os.path.join(test_img_dir, img_info['file_name'])
    image     = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    results   = model.predict(img_path, conf=CONF_THRESHOLD, verbose=False)
    boxes_obj = results[0].boxes
    overlay   = image.copy()

    for box in boxes_obj:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        score = float(box.conf[0])
        label = int(box.cls[0])
        color = tuple(int(c * 255) for c in COLORS(label)[:3])
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
        text = f'{CLASS_NAMES[label]} {score:.2f}'
        cv2.putText(overlay, text, (x1, max(y1 - 4, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

    gt_count = gt_counts.get(img_info['id'], 0)
    ax.imshow(overlay)
    ax.set_title(f'GT:{gt_count} Pred:{len(boxes_obj)}', fontsize=9)
    ax.axis('off')

plt.suptitle('RT-DETR Predictions on Test Set (conf>=0.5)', fontsize=13)
plt.tight_layout()
viz_path = os.path.join(OUTPUT_DIR, 'rtdetr_predictions.png')
plt.savefig(viz_path, dpi=150)
plt.show()
print(f'Saved: {viz_path}')

## Algorithm 5: Watershed Post-Processing

RT-DETR provides bounding boxes with class labels. Watershed refines each box crop into a pixel-precise segmentation mask.

```
Trained RT-DETR → boxes + class labels
  → For each box: crop image (+ padding) → watershed_on_crop()
  → Place mask back on full-image canvas
  → Evaluate: box mAP + mask mAP + binary IoU + counting
```

In [ ]:
# ── Watershed helpers (same as Algorithm 3 & 4) ───────────────────────────────
import cv2
import numpy as np
from collections import defaultdict
from tqdm import tqdm
from pycocotools import mask as coco_mask_util
from pycocotools.coco import COCO as _COCO
from pycocotools.cocoeval import COCOeval as _COCOeval


def watershed_on_crop(crop, padding=10):
    h, w     = crop.shape[:2]
    gray     = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    filtered = cv2.bilateralFilter(crop, 9, 75, 75)
    a_chan   = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)[:, :, 1]

    _, thresh_otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh_a = cv2.adaptiveThreshold(a_chan, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, 15, 2)
    edges    = cv2.Canny(gray, 30, 100)
    combined = cv2.bitwise_or(thresh_otsu, thresh_a)
    combined = cv2.add(combined, cv2.dilate(edges, np.ones((2, 2), np.uint8)))
    _, combined = cv2.threshold(combined, 127, 255, cv2.THRESH_BINARY)

    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    cleaned = cv2.morphologyEx(combined, cv2.MORPH_OPEN,  kernel, iterations=1)
    cleaned = cv2.morphologyEx(cleaned,  cv2.MORPH_CLOSE, kernel, iterations=2)

    bw = max(3, min(padding // 2, 8))
    border_mask = np.ones((h, w), dtype=np.uint8) * 255
    border_mask[:bw, :] = border_mask[-bw:, :] = 0
    border_mask[:, :bw] = border_mask[:, -bw:] = 0
    sure_bg = cv2.bitwise_or(cv2.dilate(cleaned, kernel, iterations=3), border_mask)

    dist = cv2.distanceTransform(cleaned, cv2.DIST_L2, 5)
    if dist.max() > 0:
        _, sure_fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
        sure_fg = np.uint8(sure_fg)
    else:
        sure_fg = np.zeros((h, w), dtype=np.uint8)
        cx, cy  = w // 2, h // 2
        r = min(cx, cy, w - cx, h - cy) // 2
        if r > 0:
            cv2.circle(sure_fg, (cx, cy), r, 255, -1)

    unknown  = cv2.subtract(sure_bg, sure_fg)
    _, marks = cv2.connectedComponents(sure_fg)
    marks    = marks + 1
    marks[unknown == 255] = 0
    marks    = cv2.watershed(crop, marks)

    mask  = np.zeros((h, w), dtype=np.uint8)
    valid = set(np.unique(marks)) - {-1, 1}
    if valid:
        best = max(valid, key=lambda lbl: np.sum(marks == lbl))
        mask[marks == best] = 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    else:
        mask = cleaned
    return mask


def apply_watershed_to_boxes(image, boxes, labels, scores, conf=0.25, padding=15):
    h, w = image.shape[:2]
    dets = []
    for box, label, score in zip(boxes, labels, scores):
        if score < conf:
            continue
        x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        x1c = max(0, x1 - padding); y1c = max(0, y1 - padding)
        x2c = min(w, x2 + padding); y2c = min(h, y2 + padding)
        if x2c - x1c < 5 or y2c - y1c < 5:
            continue
        crop_mask = watershed_on_crop(image[y1c:y2c, x1c:x2c], padding=padding)
        full_mask = np.zeros((h, w), dtype=np.uint8)
        full_mask[y1c:y2c, x1c:x2c] = crop_mask
        dets.append({'box': [x1, y1, x2, y2], 'class_id': int(label) + 1,  # YOLO 0-idx -> COCO 1-idx
                     'score': float(score), 'mask': full_mask})
    return dets


def build_gt_mask_ws(anns, img_shape):
    combined = np.zeros(img_shape[:2], dtype=np.uint8)
    for ann in anns:
        for seg in ann.get('segmentation', []):
            pts = np.array(seg, dtype=np.float32).reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(combined, [pts], 255)
    return combined


def evaluate_watershed_detections(det_by_image, coco_data, img_id_lookup):
    gt_cnts = defaultdict(int)
    gt_msks = defaultdict(list)
    for ann in coco_data['annotations']:
        gt_cnts[ann['image_id']] += 1
        gt_msks[ann['image_id']].append(ann)

    bbox_preds, segm_preds = [], []
    binary_ious, count_errs, per_image = [], [], []

    for image_id, (img_shape, fname, dets) in det_by_image.items():
        info   = img_id_lookup.get(image_id, {})
        orig_h = info.get('height', img_shape[0])
        orig_w = info.get('width',  img_shape[1])

        pred_m = np.zeros(img_shape[:2], dtype=np.uint8)
        for d in dets:
            pred_m = cv2.bitwise_or(pred_m, d['mask'])
        gt_m   = build_gt_mask_ws(gt_msks.get(image_id, []), img_shape)
        inter  = cv2.countNonZero(cv2.bitwise_and(pred_m, gt_m))
        union  = cv2.countNonZero(cv2.bitwise_or( pred_m, gt_m))
        binary_ious.append(inter / max(union, 1))

        gt_count = gt_cnts.get(image_id, 0)
        count_errs.append(abs(len(dets) - gt_count))
        per_image.append({'file': fname, 'gt_count': gt_count,
                          'pred_count': len(dets), 'count_error': abs(len(dets) - gt_count)})

        for d in dets:
            x1, y1, x2, y2 = d['box']
            bbox_preds.append({'image_id': image_id, 'category_id': d['class_id'],
                               'bbox': [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                               'score': d['score']})
            m = d['mask'].copy()
            if m.shape != (orig_h, orig_w):
                m = cv2.resize(m, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
            rle = coco_mask_util.encode(np.asfortranarray(m))
            rle['counts'] = rle['counts'].decode('utf-8')
            segm_preds.append({'image_id': image_id, 'category_id': d['class_id'],
                               'segmentation': rle, 'score': d['score']})

    gt_coco = _COCO(); gt_coco.dataset = coco_data; gt_coco.createIndex()
    b50 = b95 = m50 = m95 = 0.0
    if bbox_preds:
        dt = gt_coco.loadRes(bbox_preds)
        ev = _COCOeval(gt_coco, dt, 'bbox'); ev.evaluate(); ev.accumulate(); ev.summarize()
        b50, b95 = float(ev.stats[1]), float(ev.stats[0])
    if segm_preds:
        dt = gt_coco.loadRes(segm_preds)
        ev = _COCOeval(gt_coco, dt, 'segm'); ev.evaluate(); ev.accumulate(); ev.summarize()
        m50, m95 = float(ev.stats[1]), float(ev.stats[0])

    n = len(count_errs)
    metrics = {
        'box_map50':         b50,  'box_map50_95':       b95,
        'mask_map50':        m50,  'mask_map50_95':      m95,
        'binary_iou':        float(np.mean(binary_ious)),
        'counting_mae':      float(np.mean(count_errs)),
        'counting_within_1': float(sum(1 for e in count_errs if e <= 1) / n * 100),
        'counting_within_3': float(sum(1 for e in count_errs if e <= 3) / n * 100),
    }
    return metrics, per_image


print('Watershed helpers loaded.')

In [ ]:
CONF_WS    = 0.25
PADDING_WS = 15

# test_coco_data and test_img_dir are already defined in the inference cell above
ws_img_id_lookup = {img['id']: img for img in test_coco_data['images']}
ws_img_file_lookup = {img['file_name']: img for img in test_coco_data['images']}
ws_test_files = sorted([
    f for f in os.listdir(test_img_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

# Reload best RT-DETR weights
best_ckpt = os.path.join(OUTPUT_DIR, 'train', 'weights', 'best_rtdetr.pt')
rtdetr_ws = RTDETR(best_ckpt)
print(f'Loaded: {best_ckpt}')

det_by_image = {}
det_times    = []

for fname in tqdm(ws_test_files, desc='RTDETR+Watershed'):
    img_bgr  = cv2.imread(os.path.join(test_img_dir, fname))
    if img_bgr is None:
        continue
    img_info = ws_img_file_lookup.get(fname)
    if img_info is None:
        continue
    image_id = img_info['id']

    t0      = time.perf_counter()
    results = rtdetr_ws.predict(os.path.join(test_img_dir, fname),
                                conf=CONF_WS, verbose=False)
    det_times.append((time.perf_counter() - t0) * 1000)

    boxes_obj = results[0].boxes
    if boxes_obj and len(boxes_obj) > 0:
        boxes  = boxes_obj.xyxy.cpu().numpy()
        labels = boxes_obj.cls.cpu().numpy().astype(int)
        scores = boxes_obj.conf.cpu().numpy()
    else:
        boxes = np.zeros((0, 4)); labels = np.zeros(0, int); scores = np.zeros(0)

    dets = apply_watershed_to_boxes(img_bgr, boxes, labels, scores, CONF_WS, PADDING_WS)
    det_by_image[image_id] = (img_bgr.shape, fname, dets)

print('\nEvaluating...')
metrics, per_image = evaluate_watershed_detections(det_by_image, test_coco_data, ws_img_id_lookup)
avg_inf = float(np.mean(det_times[1:]) if len(det_times) > 1 else det_times[0])

print(f"\n{'='*50}")
print('Algorithm 5: RT-DETR + Watershed')
print(f"{'='*50}")
for k, v in metrics.items():
    print(f"  {k:25s}: {v:.4f}")
print(f"  {'avg_inference_ms':25s}: {avg_inf:.1f}")

results_data = {
    'metrics': {**metrics, 'avg_inference_ms': avg_inf},
    'per_image': per_image,
}
out_path = os.path.join(OUTPUT_DIR, 'rtdetr_watershed_results.json')
with open(out_path, 'w') as f:
    json.dump(results_data, f, indent=2)
print(f'\nSaved: {out_path}')